In [0]:
%sql
SELECT * FROM workspace.gold.dim_author_association

In [0]:
%sql
SELECT * FROM workspace.gold.dim_base_ref 

In [0]:
%sql
SELECT * FROM workspace.gold.dim_date

In [0]:
%sql
SELECT * FROM workspace.gold.dim_flags

In [0]:
%sql
SELECT * FROM workspace.gold.dim_label

In [0]:
%sql
SELECT DISTINCT label_name FROM workspace.gold.dim_label ORDER BY label_name

In [0]:
%sql
SELECT * FROM gold.dim_milestone

In [0]:
%sql
SELECT * FROM gold.dim_milestone_state_history

In [0]:
%sql
SELECT DISTINCT milestone_state FROM gold.dim_milestone_state_history

In [0]:
%sql
SELECT * FROM gold.dim_repo

In [0]:
%sql
SELECT * FROM gold.dim_timestamp

In [0]:
%sql
SELECT * FROM gold.dim_user

In [0]:
%sql
SELECT * FROM gold.dim_user
WHERE user_login LIKE '%[bot]' AND user_type != 'Bot'

In [0]:
%sql
SELECT * FROM gold.dim_user WHERE user_login NOT LIKE '%[bot]' AND user_type = 'Bot'

In [0]:
%sql
SELECT DISTINCT c.author_key
FROM gold.fact_commits c
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_user u WHERE c.author_key = u.user_key
)

In [0]:
%sql
SELECT DISTINCT c.committer_key
FROM gold.fact_commits c
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_user u WHERE c.committer_key = u.user_key
)

In [0]:
%sql
SELECT r.repo_key
FROM gold.dim_repo r
WHERE NOT EXISTS (
    SELECT 1 FROM gold.fact_commits c WHERE r.repo_key = c.repo_key
)

In [0]:
%sql
SELECT c.commit_author_date_key
FROM gold.fact_commits c
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_date d WHERE d.date_key = c.commit_author_date_key
);

In [0]:
%sql
SELECT c.commit_commiter_date_key
FROM gold.fact_commits c
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_date d WHERE d.date_key = c.commit_commiter_date_key
);

In [0]:
%sql
SELECT pr_id, COUNT(*)
FROM gold.fact_pull_request
GROUP BY pr_id
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT number, COUNT(*)
FROM gold.fact_pull_request
GROUP BY number
HAVING COUNT(*) > 1
ORDER BY COUNT(*) DESC

In [0]:
%sql
SELECT repo_key, number, COUNT(*) AS n
FROM gold.fact_pull_request
GROUP BY repo_key, number
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT pr_id, COUNT(*) AS n
FROM gold.fact_pull_request
GROUP BY pr_id
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT merge_commit_sha, COUNT(*)
FROM gold.fact_pull_request
GROUP BY merge_commit_sha
ORDER BY COUNT(*) DESC

In [0]:
%sql
SELECT pr_id, merge_commit_sha, COUNT(*)
FROM gold.fact_pull_request
GROUP BY pr_id, merge_commit_sha
HAVING COUNT(*) > 1
ORDER BY COUNT(*) DESC

In [0]:
%sql
SELECT DISTINCT repo_key FROM gold.fact_pull_request

In [0]:
%sql
SELECT pr.author_key
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_user u WHERE pr.author_key = u.user_key
)

In [0]:
%sql
SELECT DISTINCT author_association_key FROM gold.fact_pull_request

In [0]:
%sql
SELECT
  SUM(CASE WHEN repo_key = -1 THEN 1 ELSE 0 END) AS repo_unknown,
  SUM(CASE WHEN author_key = -1 THEN 1 ELSE 0 END) AS author_unknown,
  SUM(CASE WHEN author_association_key = -1 THEN 1 ELSE 0 END) AS assoc_unknown,
  SUM(CASE WHEN flag_key = -1 THEN 1 ELSE 0 END) AS flag_unknown,
  SUM(CASE WHEN created_date_key = -1 THEN 1 ELSE 0 END) AS created_date_unknown,
  SUM(CASE WHEN base_ref_key = -1 THEN 1 ELSE 0 END) AS base_ref_unknown,
  SUM(CASE WHEN milestone_key = -1 THEN 1 ELSE 0 END) AS milestone_unknown
FROM gold.fact_pull_request;

In [0]:
%sql
SELECT * FROM gold.dim_flags

In [0]:
%sql
SELECT pr.flag_key
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_flags f WHERE pr.flag_key = f.flag_key
)

In [0]:
%sql
SELECT 'created_date_key' AS col, COUNT(*) AS orphans
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_date d WHERE pr.created_date_key = d.date_key)
UNION ALL
SELECT 'updated_date_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_date d WHERE pr.updated_date_key = d.date_key)
UNION ALL
SELECT 'closed_date_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_date d WHERE pr.closed_date_key = d.date_key)
UNION ALL
SELECT 'merged_date_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_date d WHERE pr.merged_date_key = d.date_key);

In [0]:
%sql
SELECT 'merged' AS col, COUNT(*) AS out_of_range
FROM gold.fact_pull_request
WHERE merged_date_key = -1 AND time_to_merge_hours IS NOT NULL
UNION ALL
SELECT 'closed', COUNT(*)
FROM gold.fact_pull_request
WHERE closed_date_key = -1 AND flag_key IN (
    SELECT flag_key FROM gold.dim_flags WHERE state = 'closed'
);

In [0]:
%sql
SELECT MIN(date_key) AS first_day, MAX(date_key) AS last_day
FROM gold.dim_date WHERE date_key != -1;

In [0]:
%sql
SELECT 'created_timestamp_key' AS col, COUNT(*) AS orphans
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.created_timestamp_key = t.timestamp_id)
UNION ALL
SELECT 'updated_timestamp_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.updated_timestamp_key = t.timestamp_id)
UNION ALL
SELECT 'closed_timestamp_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.closed_timestamp_key = t.timestamp_id)
UNION ALL
SELECT 'merged_timestamp_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.merged_timestamp_key = t.timestamp_id);

In [0]:
%sql
SELECT 'created_timestamp_key' AS col, COUNT(*) AS orphans
FROM gold.fact_pull_request pr
WHERE pr.created_timestamp_key IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.created_timestamp_key = t.timestamp_id)
UNION ALL
SELECT 'updated_timestamp_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE pr.updated_timestamp_key IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.updated_timestamp_key = t.timestamp_id)
UNION ALL
SELECT 'closed_timestamp_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE pr.closed_timestamp_key IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.closed_timestamp_key = t.timestamp_id)
UNION ALL
SELECT 'merged_timestamp_key', COUNT(*)
FROM gold.fact_pull_request pr
WHERE pr.merged_timestamp_key IS NOT NULL
  AND NOT EXISTS (SELECT 1 FROM gold.dim_timestamp t WHERE pr.merged_timestamp_key = t.timestamp_id);

In [0]:
%sql
SELECT
  SUM(CASE WHEN created_timestamp_key = -1 THEN 1 ELSE 0 END) AS created_ts_unknown,
  SUM(CASE WHEN updated_timestamp_key = -1 THEN 1 ELSE 0 END) AS updated_ts_unknown,
  SUM(CASE WHEN created_timestamp_key IS NULL THEN 1 ELSE 0 END) AS created_ts_null,
  SUM(CASE WHEN updated_timestamp_key IS NULL THEN 1 ELSE 0 END) AS updated_ts_null
FROM gold.fact_pull_request;

In [0]:
%sql
SELECT
  SUM(CASE WHEN (closed_date_key IS NULL) != (closed_timestamp_key IS NULL) THEN 1 ELSE 0 END) AS closed_mismatch,
  SUM(CASE WHEN (merged_date_key IS NULL) != (merged_timestamp_key IS NULL) THEN 1 ELSE 0 END) AS merged_mismatch,
  SUM(CASE WHEN closed_timestamp_key IS NULL THEN 1 ELSE 0 END) AS closed_ts_null,
  SUM(CASE WHEN merged_timestamp_key IS NULL THEN 1 ELSE 0 END) AS merged_ts_null
FROM gold.fact_pull_request;

In [0]:
%sql
SELECT pr.milestone_key 
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_milestone m WHERE pr.milestone_key = m.milestone_key
)

In [0]:
%sql
SELECT COUNT(*) AS bad_milestone
FROM gold.fact_pull_request f
JOIN (
  SELECT pr_id, milestone_id FROM silver.aspnetcore_pull_requests
  UNION ALL SELECT pr_id, milestone_id FROM silver.powershell_pull_requests
  UNION ALL SELECT pr_id, milestone_id FROM silver.powertoys_pull_requests
  UNION ALL SELECT pr_id, milestone_id FROM silver.runtime_pull_requests
  UNION ALL SELECT pr_id, milestone_id FROM silver.terminal_pull_requests
  UNION ALL SELECT pr_id, milestone_id FROM silver.typescript_pull_requests
  UNION ALL SELECT pr_id, milestone_id FROM silver.vscode_pull_requests
) s ON f.pr_id = s.pr_id
WHERE f.milestone_key = -1 AND s.milestone_id IS NOT NULL;

In [0]:
%sql
SELECT * FROM gold.dim_base_ref

In [0]:
%sql
SELECT pr.base_ref_key 
FROM gold.fact_pull_request pr
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_base_ref br WHERE pr.base_ref_key = br.base_ref_key
)

In [0]:
%sql
SELECT DISTINCT is_external
FROM gold.fact_pull_request

In [0]:
%sql
SELECT COUNT(is_external) AS external_known_count
FROM gold.fact_pull_request

In [0]:
%sql
SELECT time_to_merge_hours 
FROM gold.fact_pull_request
WHERE time_to_merge_hours <= 0

In [0]:
%sql
SELECT
  MIN(assignees_count) AS min_assignees, MAX(assignees_count) AS max_assignees,
  SUM(CASE WHEN assignees_count IS NULL OR assignees_count < 0 THEN 1 ELSE 0 END) AS bad_assignees,
  MIN(reviewers_count) AS min_reviewers, MAX(reviewers_count) AS max_reviewers,
  SUM(CASE WHEN reviewers_count IS NULL OR reviewers_count < 0 THEN 1 ELSE 0 END) AS bad_reviewers,
  MIN(labels_count) AS min_labels, MAX(labels_count) AS max_labels,
  SUM(CASE WHEN labels_count IS NULL OR labels_count < 0 THEN 1 ELSE 0 END) AS bad_labels
FROM gold.fact_pull_request;

In [0]:
%sql
SELECT 'assignees_count' AS col, COUNT(*) AS mismatches
FROM gold.fact_pull_request f
LEFT JOIN (
  SELECT pr_id, COUNT(DISTINCT assignee_key) AS c
  FROM gold.pr_assignee WHERE assignee_key != -1 GROUP BY pr_id
) b ON f.pr_id = b.pr_id
WHERE f.assignees_count != COALESCE(b.c, 0)
UNION ALL
SELECT 'reviewers_count', COUNT(*)
FROM gold.fact_pull_request f
LEFT JOIN (
  SELECT pr_id, COUNT(DISTINCT reviewer_key) AS c
  FROM gold.pr_reviewer WHERE reviewer_key != -1 GROUP BY pr_id
) b ON f.pr_id = b.pr_id
WHERE f.reviewers_count != COALESCE(b.c, 0)
UNION ALL
SELECT 'labels_count', COUNT(*)
FROM gold.fact_pull_request f
LEFT JOIN (
  SELECT pr_id, COUNT(DISTINCT label_key) AS c
  FROM gold.pr_label WHERE label_key != -1 GROUP BY pr_id
) b ON f.pr_id = b.pr_id
WHERE f.labels_count != COALESCE(b.c, 0);

In [0]:
%sql
SELECT 'pr_assignee' AS bridge, COUNT(*) AS orphan_pr_ids
FROM gold.pr_assignee b
WHERE NOT EXISTS (SELECT 1 FROM gold.fact_pull_request f WHERE f.pr_id = b.pr_id)
UNION ALL
SELECT 'pr_reviewer', COUNT(*)
FROM gold.pr_reviewer b
WHERE NOT EXISTS (SELECT 1 FROM gold.fact_pull_request f WHERE f.pr_id = b.pr_id)
UNION ALL
SELECT 'pr_label', COUNT(*)
FROM gold.pr_label b
WHERE NOT EXISTS (SELECT 1 FROM gold.fact_pull_request f WHERE f.pr_id = b.pr_id);

In [0]:
%sql
SELECT
  (SELECT SUM(assignees_count) FROM gold.fact_pull_request) AS fact_assignees,
  (SELECT COUNT(*) FROM (SELECT DISTINCT pr_id, assignee_key FROM gold.pr_assignee WHERE assignee_key != -1)) AS bridge_assignees,
  (SELECT SUM(reviewers_count) FROM gold.fact_pull_request) AS fact_reviewers,
  (SELECT COUNT(*) FROM (SELECT DISTINCT pr_id, reviewer_key FROM gold.pr_reviewer WHERE reviewer_key != -1)) AS bridge_reviewers,
  (SELECT SUM(labels_count) FROM gold.fact_pull_request) AS fact_labels,
  (SELECT COUNT(*) FROM (SELECT DISTINCT pr_id, label_key FROM gold.pr_label WHERE label_key != -1)) AS bridge_labels;

In [0]:
%sql
SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT pr_id) AS distinct_pr_id
FROM gold.fact_pull_request;

In [0]:
%sql
SELECT ass.assignee_key
FROM gold.pr_assignee ass
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_user u WHERE ass.assignee_key = u.user_key
)

In [0]:
%sql
SELECT * FROM gold.pr_assignee WHERE assignee_key = -1

In [0]:
%sql
SELECT pr_id, assignee_key, COUNT(*)
FROM gold.pr_assignee
GROUP BY pr_id, assignee_key
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT pa.pr_id
FROM gold.pr_assignee pa
WHERE NOT EXISTS (
    SELECT 1 FROM gold.fact_pull_request fp WHERE fp.pr_id = pa.pr_id
)

In [0]:
%sql
SELECT l.label_key
FROM gold.pr_label l
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_label la WHERE l.label_key = la.label_key
)

In [0]:
%sql
SELECT * FROM gold.pr_label WHERE label_key = -1

In [0]:
%sql
SELECT pr_id, label_key, COUNT(*)
FROM gold.pr_label
GROUP BY pr_id, label_key
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT pl.pr_id
FROM gold.pr_label pl
WHERE NOT EXISTS (
    SELECT 1 FROM gold.fact_pull_request fp WHERE fp.pr_id = pl.pr_id
)

In [0]:
%sql
SELECT r.reviewer_key
FROM gold.pr_reviewer r
WHERE NOT EXISTS (
    SELECT 1 FROM gold.dim_user u WHERE r.reviewer_key = u.user_key
)

In [0]:
%sql
SELECT * FROM gold.pr_reviewer WHERE reviewer_key = -1

In [0]:
%sql
SELECT pr_id, reviewer_key, COUNT(*)
FROM gold.pr_reviewer
GROUP BY pr_id, reviewer_key
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT prr.pr_id
FROM gold.pr_reviewer prr
WHERE NOT EXISTS (
    SELECT 1 FROM gold.fact_pull_request fp WHERE fp.pr_id = prr.pr_id
)